# מפגש 23 — מעבדת תלמיד עם השלמות
עוגן המקור: flipper_length_mm, body_mass_g, island; seed17. השלימו ??? לפני הרצה.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
df = pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
assert len(df)==344

## 1 — בחירת תכונות וחלוקה בלי זליגה

In [ ]:
numeric=???
categorical=["island"]
X=df[numeric+categorical].copy()
y=df["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=???,stratify=y)

## 2 — ColumnTransformer בתוך Pipeline; השלימו רכיבי החיפוש

In [ ]:
numeric_pipe=Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())])
categorical_pipe=Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
prepare=ColumnTransformer([("numeric",numeric_pipe,numeric),("categorical",categorical_pipe,categorical)])
pipe=Pipeline([("prepare",prepare),("knn",KNeighborsClassifier())])
param_grid={"knn__n_neighbors":???}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
search=GridSearchCV(pipe,param_grid,scoring=???,cv=cv,refit=True,return_train_score=True)
search.fit(???,???)

## 3 — הצגת טבלת התוצאות ובחירת היפר־פרמטר, ללא Test

In [ ]:
table=pd.DataFrame(search.cv_results_)
print(table[["param_knn__n_neighbors","mean_train_score","mean_test_score","std_test_score","rank_test_score"]])
print("Chosen:",search.???)

## 4 — השוואת DummyClassifier והערכה אחת ל־Test

In [ ]:
baseline=DummyClassifier(strategy="most_frequent")
baseline_cv=cross_val_score(baseline,X_train,y_train,cv=cv,scoring="f1_macro")
baseline.fit(X_train,y_train)
prediction=search.predict(???)
baseline_prediction=baseline.predict(???)
print(f1_score(y_test,prediction,average="macro"))
print(f1_score(y_test,baseline_prediction,average="macro"))

## 5 — מסקנה מנומקת
פרטו מדוע mean_test_score אינו Outer Test, מה תפקיד refit ואיזו מגבלה קיימת לשימוש בנתוני Penguins מוכרים.

## שלב 0 — חישוב ידני והגנהלפני החיפוש: כמה התאמות CV מתקבלות משלושה ערכי K וחמישה קיפולים? הסבירו מה עושה `refit=True`. אל תציגו את תוצאת החיפוש של תלמידים אחרים.

In [ ]:
candidate_count=???fold_count=???cv_fits=???*???print("CV fits:",cv_fits)

## שלב 1 — בדיקות עצמאיות לחלוקהבדקו שהיעד אינו נמצא ב־X, ושיש הפרדה בין Train לבין Test.

In [ ]:
assert "species" not in X_train.columnsassert len(X_test)==???assert len(X_train)+len(X_test)==len(X)assert set(X_train.index).isdisjoint(???)

## שלב 2 — הבנת קיפולים והימנעות מזליגההשלימו את מספר הקיפולים; הסבירו מי מלמד את החציון/התקנון ולמה אין לבצע `prepare.fit(X_train)` לפני החיפוש.

In [ ]:
seen=np.zeros(len(X_train),dtype=int)for tr,va in cv.split(X_train,y_train):    assert set(tr).isdisjoint(set(va))    seen[va]+=1assert np.all(seen==???)

## שלב 3 — קריאת cv_results_מה ההבדל בין `mean_train_score`, `mean_test_score`, `std_test_score`, `rank_test_score`? מהי מגבלת הפירוש של פיזור חמשת הקיפולים?

In [ ]:
assert len(table)==???assert table["rank_test_score"].min()==???assert table["mean_test_score"].between(???,???).all()assert search.best_params_[???] in [3,5,9]print("Best CV macro-F1:",search.best_score_)

## שלב 4 — בדיקות Test וקו בסיסלאחר קיבוע K בלבד, השוו ל־DummyClassifier באותו macro-F1. כתבו מגבלה של Penguins, בלי לכוון את המודל שוב לפי Test.

In [ ]:
assert len(prediction)==len(???)assert len(baseline_prediction)==len(???)model_score=f1_score(y_test,prediction,average=???)baseline_score=f1_score(y_test,baseline_prediction,average=???)print(model_score,baseline_score)

## כרטיס יציאההסבירו למה `mean_test_score` מתייחס לאימות פנימי, מהו תפקיד `refit`, מדוע island דורש OneHotEncoder ולא קוד מספרי סדרתי, ומה יקרה אם נשנה K אחרי בדיקת Test.